# Plot time lag when SIC>15% in satellite data

In [1]:
## import required packages
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean
from pyproj import Proj, Transformer
import glob
import os
import string
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

In [2]:
from dask.distributed import Client

client = Client("tcp://127.0.0.1:36413")
client

<Client: 'tcp://127.0.0.1:36413' processes=8 threads=32, memory=123.95 GiB>

In [8]:
years = np.arange(2014,2021,1)

In [4]:
sic_dir = "/home/jpluser/efs-mount-point/mzahn/satellite_data/sic_nsidc_cdr"
# sic_file_paths_all = sorted(glob.glob(os.path.join(sic_dir, f"*{year}*.nc")))
sic_file_paths_all = sorted(glob.glob(os.path.join(sic_dir, "*.nc")))

In [ ]:
sic_file_paths_all

In [12]:
def find_ice_onset_doy(sic, threshold=15, duration=5):
    # Convert to binary mask: 1 if SIC >= threshold, else 0
    ice_mask = sic >= threshold

    # Rolling sum over time to find 3-day periods over threshold
    ice_rolling = ice_mask.rolling(time=duration, center=False).sum()

    # First time index where the rolling sum equals duration (i.e., 3 consecutive days over threshold)
    condition_met = (ice_rolling >= duration)

    # Find the first time index this condition is True
    def first_true_doy(x, time_coords):
        if not np.any(x):
            return np.nan
        first_index = np.argmax(x)
        return pd.to_datetime(time_coords[first_index]).dayofyear

    # Apply over grid cells
    doy = xr.apply_ufunc(
        first_true_doy,
        condition_met,
        condition_met['time'],
        input_core_dims=[['time'], ['time']],
        output_core_dims=[[]],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float]
    )

    return doy